# 📝 Exercise: Agentic RAG Workshop (4 Hours)
## Agentic RAG: From Zero to Hero

---

### 📋 Instructions

1. **Complete this by yourself** — do not use AI to help write code
2. **No copying** — each person's data will be **different** (generated from the student ID)
3. **Submit this notebook** together with the executed results (.ipynb)
4. **Score**: 10 points

> ⚠️ **The system will detect plagiarism** from embedding values, scores, and agent responses that must match the student ID

## 📦 Install Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ Installation complete!')

## 🎓 Fill in Student Information

In [ ]:
# ─── Fill in your information ───
STUDENT_NAME = ''   # e.g. 'Somchai Jaidee'
STUDENT_ID   = ''   # e.g. '6512345678'
PHONE        = ''   # e.g. '081-234-5678'
LINE_ID      = ''   # e.g. 'somchai.j'

# ─── Validation (do not edit) ───
assert len(STUDENT_ID) >= 5, '❌ Please enter your student ID!'
assert len(STUDENT_NAME) >= 2, '❌ Please enter your full name!'

print(f'✅ Full name: {STUDENT_NAME}')
print(f'✅ Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Create Personalized Dataset (Do not edit this cell)

In [ ]:
%%time
# ===== Do not edit this cell =====
# Create a personalized dataset from the student ID

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'Machine Learning is a subfield of artificial intelligence that focuses on developing algorithms that can learn from data and automatically improve performance.',
    'Deep Learning is a Machine Learning technique that uses multilayer neural networks to process complex data such as image recognition and language translation.',
    'Natural Language Processing, or NLP, is the field that enables computers to understand, interpret, and generate human language, including sentiment analysis and text summarization.',
    'Retrieval Augmented Generation, or RAG, is a technique that combines information retrieval with LLM text generation to produce accurate answers with traceable sources.',
    'A Vector Database is a database designed to store and search data in the form of embedding vectors, enabling fast search for semantically similar information.',
    'Text Embedding is the process of converting text into a set of numerical vectors that represent its semantic meaning, making it possible to compare similarity between texts.',
    'Transformer is a neural network architecture that uses the Attention mechanism to process data and is the foundation of GPT, BERT, and Gemini.',
    'Prompt Engineering is the practice of designing prompts for LLMs to achieve the desired output. Writing good prompts can greatly improve answer quality.',
    'Chunking is the process of dividing long documents into smaller pieces suitable for embedding generation. There are several methods such as Fixed-size, Recursive, and Semantic.',
    'Cosine Similarity is a method for measuring the similarity between two vectors by looking at the angle between them. A value of 1 means the same direction, and it is commonly used in NLP and Information Retrieval.',
    'An Agent is an AI system that can make decisions and use tools on its own, unlike a Chatbot that can only do scripted question-answering.',
    'Google ADK, or Agent Development Kit, is a framework for building AI Agents with Python. It supports Multi-Agent systems and Tool Calling, and works well with Gemini.',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# Create a personalized query
all_queries = [
    'Techniques for retrieving semantically similar information',
    'Methods for splitting documents into smaller parts',
    'Using AI to make decisions and call tools',
    'Converting text into numbers for comparison',
    'Techniques for generating answers from retrieved information',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ Created personalized data for {STUDENT_ID}')
print(f'📁 Number of files: {len(selected)} files')
print(f'🔍 Personalized query: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} characters)')

---
## 🎯 Step 1: Chunk + Embed + Search (3 points)

- Combine text from all files in `homework_data/`
- Chunk using `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
- Create embeddings using `intfloat/multilingual-e5-large`
- Search using the query: `MY_QUERY` (generated from the student ID)
- Store in a Qdrant collection named `f'hw_{STUDENT_ID}'`

**📝 Report:**
1. How many chunks are there in total?
2. Which chunk has the highest similarity? (score with 4 decimal places)
3. What are the scores of the Top-3 results from Qdrant?

In [ ]:
# Step 1: Write your code here

# 💡 Hint:
#   1. Read files from 'homework_data/' and combine into one text
#   2. from langchain_text_splitters import RecursiveCharacterTextSplitter
#   3. splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
#   4. chunks = splitter.split_text(all_text)
#   5. from sentence_transformers import SentenceTransformer
#   6. model = SentenceTransformer('intfloat/multilingual-e5-large')
#   7. passages = ['passage: ' + c for c in chunks]
#   8. embeddings = model.encode(passages)
#   9. query_emb = model.encode(f'query: {MY_QUERY}')
#  10. Use cosine_similarity() to find the most similar chunk
#  11. from qdrant_client import QdrantClient, models
#  12. Create collection, upsert, query_points



# ✅ Self-check (uncomment after finishing your code)
# assert len(chunks) > 0, '❌ Chunking has not been done yet'
# assert len(qdrant_results) == 3, '❌ Should get top_k=3 from Qdrant'
# print(f'✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

---
## 🎯 Step 2: Agent + Custom Tool (3 points)

- Set up Gemini API Key (Colab Secrets)
- Create at least 1 **Custom Tool** (must not duplicate the BMI example from class)
- Create an **Agent** with Google ADK that can use the Tool
- Test the conversation → show that the Agent can actually call the Tool

**📝 Report:**
1. What does your Tool do? (Explain in 1-2 sentences)
2. Show the output where the Agent successfully calls the Tool
3. Why is the docstring important? (Explain in 1-2 sentences)

In [ ]:
# Step 2: Fill in the code below

# Set API Key
import os
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 Paste API Key: ')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# ─── Create a Custom Tool (Do not use BMI) ───
# 💡 Examples: temperature conversion, GPA calculation, interest calculation, etc.
def my_custom_tool(param1: float, param2: float) -> str:
    """Explain what this tool does (very important! The Agent will read this docstring)

    Args:
        param1: explain parameter 1
        param2: explain parameter 2
    """
    # TODO: write the logic here
    result = 0  # change this
    return f'Result: {result}'

tool = FunctionTool(my_custom_tool)

# ─── Create Agent ───
my_agent = LlmAgent(
    name='my_assistant',
    model='gemini-2.5-flash',
    instruction='You are an assistant... (edit this to match your tool)',
    tools=[tool]
)

# ─── Test ───
async def chat_with_agent(agent, message):
    runner = InMemoryRunner(agent=agent, app_name='homework')
    session = await runner.session_service.create_session(
        app_name='homework', user_id='student'
    )
    content = genai_types.Content(
        role='user', parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# Ask a question that requires your tool
answer = await chat_with_agent(my_agent, 'Enter a question that requires your tool')
print(f'🤖 Agent: {answer}')

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent did not respond'
print(f'✅ Step 2 passed!')

---
## 🎯 Step 3: RAG Agent + Quality Evaluation (4 points)

- Create a **RAG Tool** that searches from Qdrant (use the collection from Step 1)
- Create a **RAG Agent** that uses the RAG Tool to answer questions
- Ask 3 required questions → save the answers
- Score the answers with **LLM-as-Judge** (use Gemini to score 1-5)

**Required questions:**
```python
questions = [
    f'query: {MY_QUERY}',   # personalized query
    'What is Embedding?',
    'Why is RAG important?'
]
```

**📝 Report:**
1. The RAG Agent's answers to all 3 questions
2. What scores did LLM-as-Judge assign? (1-5 for each question)
3. Explain: How does the Agent decide to search from Qdrant? (2-3 sentences)

In [ ]:
# Step 3: Fill in the code below

# ─── A) Create RAG Tool ───
def search_knowledge(query: str) -> str:
    """Search for information from the knowledge base stored in Qdrant
    Use this when searching for information about AI, Machine Learning, and NLP

    Args:
        query: the question or topic to search for
    """
    # TODO: use qdrant.query_points() to search from the collection created in Step 1
    # results = qdrant.query_points(collection_name=..., query=..., limit=3)
    # return the results as a string
    pass

# ─── B) Create RAG Agent ───
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-2.5-flash',
    instruction='You are an AI that answers questions from a knowledge base. Always use the search_knowledge tool to search for information before answering. Answer in Thai.',
    tools=[FunctionTool(search_knowledge)]
)

# ─── C) Ask 3 questions ───
questions = [
    MY_QUERY,  # personalized query
    'What is Embedding?',
    'Why is RAG important?'
]

rag_answers = []
for q in questions:
    ans = await chat_with_agent(rag_agent, q)
    rag_answers.append({'question': q, 'answer': ans})
    print(f'\n{"="*50}')
    print(f'❓ {q}')
    print(f'🤖 {ans}')

# ─── D) LLM-as-Judge ───
from google import genai
judge_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

# 📋 Template for judge (you can use this as is)
JUDGE_PROMPT = '''You are an AI answer quality evaluator

Question: {question}
Answer: {answer}

Give a score from 1-5 based on the criteria:
- 5 = Correct, complete, and clearly explained
- 4 = Correct, but missing some details
- 3 = Partially correct, with minor errors
- 2 = Off-topic or contains several mistakes
- 1 = Completely wrong or no answer

Respond in JSON: {{"score": 0, "reason": "..."}}
'''

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
for qa in rag_answers:
    prompt = JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer'])
    resp = judge_client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
    )
    result = json.loads(resp.text)
    print(f'  ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')

# ✅ Self-check
assert len(rag_answers) == 3, '❌ Must answer all 3 questions'
print(f'\n✅ Step 3 passed: answered all 3 questions + completed LLM-as-Judge!')

## 📊 Grading Criteria

| Step | Points | Criteria |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline works, Qdrant results are correct |
| 2. Agent + Custom Tool | 3 | Tool works, Agent can call it, explains docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent answers all 3 questions, LLM-as-Judge scores them, explanation provided |
| **Total** | **10** | |

---
## ✅ Verify Answers

Run the cell below to generate a **Verification Code** for submission

In [ ]:
# ===== Do not edit this cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 Full name: {STUDENT_NAME}')
print(f'🎓 Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 Submit before: _________________ (set by instructor)')
print('=' * 50)
print()
print('📋 Checklist before submission:')
print('  [ ] Completed all personal information')
print('  [ ] Step 1: Chunk + Embed + Qdrant works')
print('  [ ] Step 2: Agent + Tool works')
print('  [ ] Step 3: RAG Agent answered 3 questions + LLM-as-Judge')
print('  [ ] All cells have been run and contain outputs')